<a href="https://colab.research.google.com/github/malakmostef/BigData/blob/main/TP01_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP01: Python Web Scraping Case-study
**Module:** Big Data Analytics — M2 IA & Data Science — جامعة الوادي 2026/2027

**الهدف:** جلب صفحات موقع تعليمي، استخراج الاقتباسات (Quotes)، حفظها في ملف CSV بواسطة pandas، والتأكد أن الملف يحتوي على **1000 سطر على الأقل**.

**خطة العمل:**
1. جلب كل صفحات `quotes.toscrape.com` (موقع مخصص للتدريب على الـ scraping) وعرض كود HTML.
2. استخراج الاقتباسات بـ BeautifulSoup.
3. هذا الموقع يحتوي على **100 اقتباس فقط** (10 صفحات × 10)، لذلك نضيف مصدراً ثانياً (Goodreads) للوصول إلى 1000+.
4. الحفظ في CSV والتحقق من الشرط.

## 0) المكتبات والإعدادات

In [1]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# ترويسة User-Agent حتى لا يرفض الموقع الطلب
HEADERS = {"User-Agent": "Mozilla/5.0 (TP01 educational scraping - University of El Oued)"}

# عدد الأحرف المعروضة من كل صفحة HTML (ضع None لعرض الكود كاملاً)
PREVIEW = 1500

def get_html(url, retries=3, timeout=15):
    """تجلب محتوى صفحة مع إعادة المحاولة عند الفشل."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if r.status_code == 200:
                r.encoding = "utf-8"
                return r.text
            print(f"[!] {url} -> HTTP {r.status_code} (محاولة {attempt})")
        except requests.RequestException as e:
            print(f"[!] خطأ في الاتصال بـ {url}: {e} (محاولة {attempt})")
        time.sleep(2)
    return None

## السؤال 1: جلب كل صفحات الموقع وعرض كود HTML
نبدأ من الصفحة الأولى ونتبع رابط **Next** (`li.next > a`) حتى لا يبقى رابط تالٍ.

In [2]:
BASE = "https://quotes.toscrape.com"
pages_html = {}          # {رقم الصفحة: كود HTML}

url, page = BASE + "/", 1
while url:
    html = get_html(url)
    if html is None:
        print("توقف الجلب: تعذّر تحميل", url)
        break
    pages_html[page] = html

    print(f"\n===== الصفحة {page} | {url} | {len(html)} حرف =====")
    print(html if PREVIEW is None else html[:PREVIEW] + "\n...[تم الاقتطاع]...")

    soup = BeautifulSoup(html, "html.parser")
    nxt = soup.select_one("li.next > a")          # رابط الصفحة التالية
    url = BASE + nxt["href"] if nxt else None
    page += 1
    time.sleep(0.5)                                # تأخير بسيط احتراماً للخادم

print(f"\nعدد الصفحات المجلوبة: {len(pages_html)}")


===== الصفحة 1 | https://quotes.toscrape.com/ | 11021 حرف =====
<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div class="col-md-4">
                <p>
                
                    <a href="/login">Login</a>
                
                </p>
            </div>
        </div>
    

<div class="row">
    <div class="col-md-8">

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without c

## السؤال 2: استخراج الاقتباسات بـ BeautifulSoup
كل اقتباس داخل `div.quote` ويحتوي على: النص (`span.text`)، المؤلف (`small.author`)، والوسوم (`a.tag`).

In [3]:
def parse_toscrape(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for q in soup.select("div.quote"):
        rows.append({
            "quote":  q.select_one("span.text").get_text(strip=True).strip("\u201c\u201d\" "),
            "author": q.select_one("small.author").get_text(strip=True),
            "tags":   ", ".join(t.get_text(strip=True) for t in q.select("a.tag")),
            "source": "quotes.toscrape.com",
        })
    return rows

all_rows = []
for p, html in pages_html.items():
    rows = parse_toscrape(html)
    all_rows.extend(rows)
    print(f"الصفحة {p}: {len(rows)} اقتباس")

print(f"\nالمجموع من quotes.toscrape.com: {len(all_rows)} اقتباس")
pd.DataFrame(all_rows).head()

الصفحة 1: 10 اقتباس
الصفحة 2: 10 اقتباس
الصفحة 3: 10 اقتباس
الصفحة 4: 10 اقتباس
الصفحة 5: 10 اقتباس
الصفحة 6: 10 اقتباس
الصفحة 7: 10 اقتباس
الصفحة 8: 10 اقتباس
الصفحة 9: 10 اقتباس
الصفحة 10: 10 اقتباس

المجموع من quotes.toscrape.com: 100 اقتباس


,quote,author,tags,source
0,The world as we have created it is a process o...,Albert Einstein,"change, deep-thoughts, thinking, world",quotes.toscrape.com
1,"It is our choices, Harry, that show what we tr...",J.K. Rowling,"abilities, choices",quotes.toscrape.com
2,There are only two ways to live your life. One...,Albert Einstein,"inspirational, life, live, miracle, miracles",quotes.toscrape.com
3,"The person, be it gentleman or lady, who has n...",Jane Austen,"aliteracy, books, classic, humor",quotes.toscrape.com
4,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,"be-yourself, inspirational",quotes.toscrape.com


## إضافة مصدر ثانٍ للوصول إلى 1000+ سطر
الموقع الأول لا يتجاوز **100 اقتباس**، لذلك نجلب صفحات من `goodreads.com/quotes` (30 اقتباساً في الصفحة).
الدالة تتوقف تلقائياً عند الوصول للعدد المطلوب أو عند فشل الجلب.

In [4]:
def parse_goodreads(html):
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for q in soup.select("div.quoteDetails"):
        t = q.select_one("div.quoteText")
        if not t or not t.contents:
            continue
        text = str(t.contents[0]).strip().strip("\u201c\u201d\" ")      # أول نص قبل علامة ―
        a = q.select_one("span.authorOrTitle")
        author = a.get_text(strip=True).rstrip(",") if a else ""
        tags = ", ".join(x.get_text(strip=True) for x in q.select("div.greyText.smallText.left a"))
        if text:
            rows.append({"quote": text, "author": author, "tags": tags, "source": "goodreads.com"})
    return rows

TARGET = 1100          # هامش أمان فوق 1000 لأن بعض المكرر سيُحذف
MAX_PAGES = 60

for p in range(1, MAX_PAGES + 1):
    if len(all_rows) >= TARGET:
        break
    html = get_html(f"https://www.goodreads.com/quotes?page={p}")
    if html is None:
        print("توقف: تعذّر جلب الصفحة", p)
        break
    rows = parse_goodreads(html)
    if not rows:
        print(f"الصفحة {p}: لا اقتباسات (ربما تغيّر هيكل الموقع أو تم الحظر) -> توقف")
        break
    all_rows.extend(rows)
    print(f"Goodreads صفحة {p}: +{len(rows)} | الإجمالي = {len(all_rows)}")
    time.sleep(1.5)

Goodreads صفحة 1: +30 | الإجمالي = 130
Goodreads صفحة 2: +30 | الإجمالي = 160
Goodreads صفحة 3: +29 | الإجمالي = 189
Goodreads صفحة 4: +30 | الإجمالي = 219
Goodreads صفحة 5: +30 | الإجمالي = 249
Goodreads صفحة 6: +30 | الإجمالي = 279
Goodreads صفحة 7: +30 | الإجمالي = 309
Goodreads صفحة 8: +30 | الإجمالي = 339
Goodreads صفحة 9: +30 | الإجمالي = 369
Goodreads صفحة 10: +30 | الإجمالي = 399
Goodreads صفحة 11: +30 | الإجمالي = 429
Goodreads صفحة 12: +30 | الإجمالي = 459
Goodreads صفحة 13: +30 | الإجمالي = 489
Goodreads صفحة 14: +30 | الإجمالي = 519
Goodreads صفحة 15: +30 | الإجمالي = 549
Goodreads صفحة 16: +30 | الإجمالي = 579
Goodreads صفحة 17: +30 | الإجمالي = 609
Goodreads صفحة 18: +30 | الإجمالي = 639
Goodreads صفحة 19: +30 | الإجمالي = 669
Goodreads صفحة 20: +30 | الإجمالي = 699
Goodreads صفحة 21: +30 | الإجمالي = 729
Goodreads صفحة 22: +30 | الإجمالي = 759
Goodreads صفحة 23: +30 | الإجمالي = 789
Goodreads صفحة 24: +30 | الإجمالي = 819
Goodreads صفحة 25: +30 | الإجمالي = 849
Goodreads

## السؤال 3: الحفظ في ملف CSV بواسطة pandas

In [5]:
df = pd.DataFrame(all_rows)
df = df.drop_duplicates(subset="quote").reset_index(drop=True)   # حذف المكرر
df.insert(0, "id", range(1, len(df) + 1))

# utf-8-sig حتى يظهر النص صحيحاً عند فتح الملف في Excel
df.to_csv("quotes.csv", index=False, encoding="utf-8-sig")
print("تم الحفظ في quotes.csv")
df.head(10)

تم الحفظ في quotes.csv


,id,quote,author,tags,source
0,1,The world as we have created it is a process o...,Albert Einstein,"change, deep-thoughts, thinking, world",quotes.toscrape.com
1,2,"It is our choices, Harry, that show what we tr...",J.K. Rowling,"abilities, choices",quotes.toscrape.com
2,3,There are only two ways to live your life. One...,Albert Einstein,"inspirational, life, live, miracle, miracles",quotes.toscrape.com
3,4,"The person, be it gentleman or lady, who has n...",Jane Austen,"aliteracy, books, classic, humor",quotes.toscrape.com
4,5,"Imperfection is beauty, madness is genius and ...",Marilyn Monroe,"be-yourself, inspirational",quotes.toscrape.com
5,6,Try not to become a man of success. Rather bec...,Albert Einstein,"adulthood, success, value",quotes.toscrape.com
6,7,It is better to be hated for what you are than...,André Gide,"life, love",quotes.toscrape.com
7,8,"I have not failed. I've just found 10,000 ways...",Thomas A. Edison,"edison, failure, inspirational, paraphrased",quotes.toscrape.com
8,9,A woman is like a tea bag; you never know how ...,Eleanor Roosevelt,misattributed-eleanor-roosevelt,quotes.toscrape.com
9,10,"A day without sunshine is like, you know, night.",Steve Martin,"humor, obvious, simile",quotes.toscrape.com


## السؤال 4: التحقق من 1000 سطر على الأقل

In [8]:
check = pd.read_csv("quotes.csv", encoding="utf-8-sig")
print("عدد الأسطر:", len(check))
print(check["source"].value_counts(), "\n")

if len(check) >= 1000:
    print(" الشرط محقق: الملف يحتوي على 1000 سطر أو أكثر")
else:
    print(f" ينقص {1000 - len(check)} سطر: ارفع MAX_PAGES أو TARGET وأعد تشغيل خلية Goodreads")

عدد الأسطر: 1026
source
goodreads.com          926
quotes.toscrape.com    100
Name: count, dtype: int64 

 الشرط محقق: الملف يحتوي على 1000 سطر أو أكثر
